In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Load your key safely
load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

# 2. Build the client with built-in retry options
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(
            attempts=5,                       # Maximum retry attempts
            initial_delay=1.0,                # Initial delay in seconds
            exp_base=2.0,                     # Backoff multiplier (2 is standard for binary exponential backoff)
            http_status_codes=[429, 500, 503, 504] # Retry on these specific errors
        )
    )
)

# 3. Your LLM function stays perfectly clean
def llm(prompt):
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
    )
    return response.text


In [2]:
import time
import requests

In [3]:
response_text = llm("Hey, what's up?")
print(response_text)

Hey there! Not much, just chilling and ready to chat.

What's up with you? Anything interesting happening?


In [6]:
time.sleep(5)
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That's exciting! It's great you're interested in the course.

To give you the most accurate answer, I'll need a little more information, as enrollment depends entirely on the specific course and its provider:

1.  **What is the name of the course?**
2.  **Where did you discover it (e.g., specific website, university, online platform like Coursera, edX, Udemy, a private school, etc.)?**

**In general, here are the common scenarios:**

*   **Self-Paced Courses (like many on Udemy, Coursera, edX, or independent platforms):** You can often join and start immediately after purchasing/enrolling.
*   **Cohort-Based Courses (with live sessions or a specific start/end date):** These usually have fixed enrollment periods. You might be able to join the *next* cohort, but not the current one if it's already started or full.
*   **University/Academic Courses:** These have very strict application deadlines and start dates, usually aligned with academic semesters.
*   **Waitlists:** If a course is po

In [55]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [56]:
time.sleep(20)
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with 'I don't know.'
Question :
{question}

Context:
{context}
"""


In [9]:
time.sleep(5)
answer = llm(prompt)
print(answer)

Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


FAQ Dataset

In [58]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
course_raw = response.json()

In [60]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in course_raw:
    course_url = f"""{url_prefix}{course['path']}"""
    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
len(documents)

1350

In [15]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

Search basics

In [61]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [62]:
question = "I just discovered the course. Can I join now?"
search_results = index.search(
    question,
    boost_dict= {"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results = 5
)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [63]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [65]:
def search(question,course= 'llm-zoomcamp'):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course":course}

    return index.search(question,
                        boost_dict = boost_dict,
                        filter_dict = filter_dict,
                        num_results = 5)

In [66]:
time.sleep(5)
search_results = search(question)

Building the Prompt

In [68]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [70]:
USER_PROMPT_TEMPLATE = """
Question :
{question}

Context
{context}
"""

In [71]:
def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append(doc['section'])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc['answer'])
        lines.append("")
    return "\n".join(lines).strip()

In [72]:
def build_prompt(question,search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question = question,
        context = context
    )
    return prompt.strip()

In [73]:
time.sleep(5)
prompt = build_prompt(question, search_results)

print(prompt)

Question :
I just discovered the course. Can I join now?

Context
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

The LLM

In [74]:
time.sleep(10)
response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents = prompt)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [26]:
response.text

'Yes, you can join now!\n\nThe course materials (videos and GitHub notebooks) are available, and you can start whenever you want.\n\nHowever, if you wish to receive a certificate, you need to submit your project while submissions are still open and complete the course with the "live" cohort, which includes participating in peer reviews. You don\'t need a confirmation email to start; you can simply begin learning and submitting homework.\n\nTo get started, refer to:\n*   The [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n*   The [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n*   The [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\nDeadlines for homework and project submissions are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).'

In [27]:
response.candidates[0].content.parts[0].text

'Yes, you can join now!\n\nThe course materials (videos and GitHub notebooks) are available, and you can start whenever you want.\n\nHowever, if you wish to receive a certificate, you need to submit your project while submissions are still open and complete the course with the "live" cohort, which includes participating in peer reviews. You don\'t need a confirmation email to start; you can simply begin learning and submitting homework.\n\nTo get started, refer to:\n*   The [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/)\n*   The [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/)\n*   The [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp)\n\nDeadlines for homework and project submissions are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).'

In [28]:
print(dir(response))

['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__firstlineno__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__pydantic_serializer__',

In [24]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=127,
  prompt_token_count=513,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=513
    ),
  ],
  thoughts_token_count=881,
  total_token_count=1521
)

In [25]:
# To inspect individual properties:
print("Input (Prompt) Tokens: ", response.usage_metadata.prompt_token_count)
print("Reasoning (Thinking) Tokens: ", response.usage_metadata.thoughts_token_count)
print("Output (Answer) Tokens: ", response.usage_metadata.candidates_token_count)
print("Total Cost Window Tokens: ", response.usage_metadata.total_token_count)

Input (Prompt) Tokens:  513
Reasoning (Thinking) Tokens:  881
Output (Answer) Tokens:  127
Total Cost Window Tokens:  1521


In [29]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
       response.usage_metadata.prompt_token_count * input_price +
       response.usage_metadata.candidates_token_count * output_price)
cost

0.0013792499999999998

In [75]:
time.sleep(5)
config = types.GenerateContentConfig(system_instruction = INSTRUCTIONS)
message_history = [
    {'role':"user",
     'parts':[{'text':prompt }]
    }
]
time.sleep(5)
response = client.models.generate_content(
           model='gemini-2.5-flash',
           contents = message_history,
           config = config
)
print(response)

sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text="""Yes, you can still join the course. You can start whenever you want as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted."""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-2.5-flash' prompt_feedback=None response_id='Dd44au3fNbHZjuMPztzaoQk' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=48,
  prompt_token_count=567,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=567
    ),
  ],
  thoughts_token_count=250,
  total_token_count=865
) model_status=None automatic_function_calling_history=[] parsed=None


In [76]:
# 1. Print the actual text answer clearly
print("--- ANSWER ---")
print(response.text)

# 2. Track your exact token consumption
print("--- METRICS ---")
print(f"Prompt Tokens: {response.usage_metadata.prompt_token_count}")
print(f"Candidate Tokens: {response.usage_metadata.candidates_token_count}")

--- ANSWER ---
Yes, you can still join the course. You can start whenever you want as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.
--- METRICS ---
Prompt Tokens: 567
Candidate Tokens: 48


In [78]:
def llm(instructions,user_prompt,model = 'gemini-2.5-flash'):
    config = types.GenerateContentConfig(system_instruction = instructions)
    message_history = [{'role':"user",
                        "parts":[{'text': user_prompt}]
                        }]
    response = client.models.generate_content(model = model,
                                             contents = message_history,
                                             config = config)
    return response.text

Full RAG

In [81]:
def rag(query , model= 'gemini-2.5-flash'):
    search_results = search(query)
    prompt = build_prompt(query,search_results)
    answer = llm(INSTRUCTIONS,prompt,model = model)
    return answer

In [82]:
time.sleep(10)
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. You can start whenever you want, and the videos and GitHub materials are available.

However, if you want to receive a certificate, you will need to submit your project while submissions are still being accepted.


In [83]:
a = rag("How do I get a certificate?")
print(a)

To get a certificate, you need to:

*   Finish the course with a "live" cohort. Certificates are not awarded for self-paced mode.
*   Pass the Capstone project.
*   Peer-review 3 capstone projects after submitting your own. This can only be done when the course is running.
*   Ensure your official name (as in your identification documents) is updated in the second field of your course profile, as this is the name that will appear on your certificate.


In [84]:
!uv pip install gitsource

Resolved 8 packages in 1.73s
Prepared 2 packages in 335ms
Installed 2 packages in 60ms
 + gitsource==0.0.5
 + python-frontmatter==1.3.0


72


In [88]:
documents[0]

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp